# 01 · Data agent
Documentation: `docs/agents/data_agent.md`. Run the cells in order with the `.venv` kernel.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_rows', 80); pd.set_option('display.width', 200)
D = ROOT / 'data'
def ok(p):
    if not Path(p).exists(): print('File not available yet:', p); return False
    return True

In [ ]:
uni = pd.read_parquet(D/'processed/universe.parquet')
px = pd.read_parquet(D/'processed/prices.parquet')
print('Assets in the universe:', len(uni), '· with data:', int(uni.has_data.sum()), '· eligible:', int(uni.eligible.sum()))
print('Prices from', px.index[0].date(), 'to', px.index[-1].date())
uni.groupby(['asset_class','region']).size().unstack(fill_value=0)

## Coverage over time: assets with prices, by year and class

In [ ]:
cov = px.notna().groupby(px.index.year).any().T.join(uni['asset_class']).groupby('asset_class').sum().T
cov.plot(figsize=(12,4)); plt.title('Assets with at least one price in the year'); plt.show(); cov.tail()

## Quality: missing data, years of history, tickers without data

In [ ]:
q = pd.DataFrame({'missing_pct': px.isna().mean().round(3), 'last': px.apply(lambda s: s.last_valid_index())}).join(uni[['asset_class','history_years']])
print('Without data:', uni.index[~uni.has_data].tolist())
q.sort_values('missing_pct', ascending=False).head(15)

## FX rates (units of currency per 1 EUR)

In [ ]:
fx = pd.read_parquet(D/'processed/fx.parquet')
(fx/fx.iloc[0]*100).plot(figsize=(12,4), logy=True); plt.title('FX rates, base 100'); plt.show()

## Macro FRED

In [ ]:
macro = pd.read_parquet(D/'processed/macro.parquet')
print(macro.apply(lambda s: s.first_valid_index()).astype(str))
macro[['DGS10','T10Y2Y','VIXCLS','NFCI','ECBDFR']].ffill().plot(subplots=True, figsize=(12,9)); plt.show()

## SEC fundamentals: coverage and an example

In [ ]:
f = pd.read_parquet(D/'processed/fundamentals.parquet')
print('Companies:', f.ticker.nunique(), '· rows:', len(f))
display(f.groupby('item').ticker.nunique().rename('companies'))
f.groupby(f.filed.dt.year).ticker.nunique().plot(kind='bar', figsize=(10,3), title='Companies with filings per year'); plt.show()
f[(f.ticker=='AAPL') & f.item.isin(['net_income','equity','assets'])].pivot_table(index='end', columns='item', values='val').tail(6)

## Synthetic history of capital-efficient ETFs: reconstruction vs real prices

In [ ]:
from core.config import load_config
from agents.portfolio_agent.agent import load
cfg = load_config(); recipes = cfg['synthetic_history']
pl, pe, vol, u2, rf = load(D/'processed', 'EUR')
display(u2.loc[list(recipes), ['synthetic_from']])
r = px.pct_change(fill_method=None); cash = macro['DFF'].reindex(px.index).ffill()/100/252
for t, rec in recipes.items():
    syn = sum(w*r[l] for l, w in rec['legs'].items()) + rec['cash']*cash - rec['fee']/252
    real = r[t].dropna(); s = syn.reindex(real.index)
    print(f"{t}: correlation {real.corr(s):.3f} · annual return real {real.mean()*252:+.1%} synthetic {s.mean()*252:+.1%} · tracking error {(real-s).std()*np.sqrt(252):.1%}")

In [ ]:
t = 'NTSX'; real = px[t].dropna(); syn = pl[t].loc[:real.index[0]]
ax = (pl[t]/pl[t].dropna().iloc[0]).plot(figsize=(12,4), label='series used (synthetic + real)', logy=True)
ax.axvline(real.index[0], color='grey', ls='--'); plt.title(f'{t}: the dashed line marks the ETF inception'); plt.legend(); plt.show()